# 11장 실습 — 시뮬레이션 루프 직접 짜기 (채점)

직접 짜는 셋 중 마지막입니다. 채울 파일은 노트북이 아니라 옆의 **`labs/ch11_simloop.py`** 입니다.
편집기에서 파일을 고치고 저장한 뒤 노트북 셀을 다시 실행하면 바로 반영됩니다. 첫 셀의 `autoreload` 덕분입니다.

지금까지 만든 것을 전부 이어 붙입니다.
8장의 수요가 호출을 만들고, 10장의 배차가 차를 고르고, 소요시간은 직선거리 ÷ 25km/h 로 근사합니다.
이 장에서 만드는 것은 그것들을 1분마다 부르는 바깥 루프입니다.

채점 기준은 실제 DTUMOS 엔진과의 거리입니다. 평균 대기시간이 1.5분 이내로 붙어야 합니다.
완전히 같을 수는 없습니다. 어디서 왜 갈라지는지를 설명할 수 있으면 됩니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

import ch11_simloop as sol      # 여러분이 채우는 파일

## 1. 한 스텝에 무엇을 하는가 (교재 11.1)

시간을 1분씩 밉니다. 매 분에 하는 일은 다섯 가지이고, 순서가 정해져 있습니다.

1. 이번 분에 들어온 호출을 대기 목록에 넣습니다
2. 너무 오래 기다린 호출을 포기 처리합니다
3. 대기 승객과 빈 차가 둘 다 있으면 배차합니다
4. 배차된 차의 다음 가용 시각을 계산합니다
5. 이번 분의 상태를 기록합니다

3번이 10장에서 만든 것입니다. 나머지가 이 장의 일입니다.
접수보다 배차를 먼저 하면 방금 들어온 호출이 1분 늦게 처리되므로 순서를 바꾸지 않습니다.

먼저 입력 데이터의 모양을 봅니다. `simulate` 는 이 두 DataFrame 을 받습니다.

In [ ]:
from smartmob.data import load_demand, load_vehicles

demand = load_demand("hanam")        # id, request_time, origin_lat/lon, dest_lat/lon
vehicles = load_vehicles("hanam")    # id, work_start, work_end, lat, lon

print(f"호출 {len(demand):,}건, 차량 {len(vehicles)}대")
print(f"수요 컬럼: {list(demand.columns)}")
print(f"차량 컬럼: {list(vehicles.columns)}")
vehicles.head(3)

`request_time` 과 `work_start`, `work_end` 는 자정부터 센 분입니다. 1080이 18:00 입니다.
차량마다 근무 시간이 있으므로 "80대"가 항상 80대는 아닙니다. 0장에서 본 그 현상의 원인입니다.

## 2. 작은 예제로 정답을 먼저 봅니다 (교재 11.3)

하남 전체로 가기 전에 승객 3명, 차 2대, 18:00~18:10 짜리로 시작합니다.
교재 11.3절과 같은 좌표입니다. 교재의 정돈본 `smartmob.teaching.simloop.simulate` 로 돌려 두면,
여러분의 루프가 무엇을 내야 하는지 셀 하나로 확인할 수 있습니다.
소요시간은 직선거리를 시속 25km 로 나누는 `straight_line_time` 입니다.

In [ ]:
import pandas as pd

from smartmob.teaching.simloop import simulate as reference, straight_line_time

toy_demand = pd.DataFrame({
    "id": [0, 1, 2],
    "request_time": [1080, 1080, 1083],                  # 둘은 18:00, 하나는 18:03
    "origin_lat": [37.540, 37.541, 37.545], "origin_lon": [127.200, 127.201, 127.205],
    "dest_lat":   [37.550, 37.560, 37.530], "dest_lon":   [127.210, 127.220, 127.190],
})
toy_vehicles = pd.DataFrame({
    "id": [0, 1], "work_start": [1080, 1080], "work_end": [1440, 1440],
    "lat": [37.539, 37.560], "lon": [127.199, 127.230],
})

toy_answer = reference(toy_demand, toy_vehicles, 1080, 1090, travel_time=straight_line_time)
display(toy_answer.record)
for r in toy_answer.requests:
    print(f"승객 {r.id}: 차량 {r.vehicle_id}, 배차 {r.assigned_time}, "
          f"탑승 {r.pickup_time:.2f}, 대기 {r.wait_min:.2f}분")

18:00 에 호출 둘이 들어오고 차 둘이 비어 있어 바로 배차됩니다. 그 뒤로 두 차는 계속 운행 중입니다.
18:03 에 세 번째 호출이 들어오지만 빈 차가 없어 `waiting_passenger_cnt` 가 1이 됩니다.
차량 0이 승객을 내려 주는 18:05.75 를 지나 18:06 에 배차됩니다. 배차까지 3분 기다렸습니다.
포기 기준 10분 안이라 `fail_passenger_cnt` 는 0으로 남습니다.

승객 1의 대기 8.96분은 배차는 즉시 됐지만 차량 1이 멀리서 오느라 걸린 시간입니다.
이 둘을 구분하는 것이 7절의 주제입니다.

## 3. 빈칸 채우기

`labs/ch11_simloop.py` 의 빈칸 다섯 자리를 이 순서로 채웁니다. 뒤의 것이 앞의 것을 씁니다.

`Vehicle.idle` → `build_costs` → `assign` → `simulate` → `SimResult.summary`

각 함수의 docstring 에 해야 할 일이 적혀 있습니다. 하나 채울 때마다 아래 셀로 확인합니다.
비어 있는 빈칸은 `[ ] 아직 구현하지 않았습니다` 로 표시되고 노트북은 계속 돌아갑니다.

### 3.1 `Vehicle.idle` — 지금 배차받을 수 있는가

근무 중이면서 `free_at` 이 지금 시각 이하이면 빈 차입니다. 상태 변수 없이 이 두 조건으로 끝납니다.

In [ ]:
banner("3.1 Vehicle.idle")
try:
    veh = sol.Vehicle(id=0, location=(37.539, 127.199), work_start=1080, work_end=1440, free_at=1085.75)
    expect("18:03 (운행 중)", veh.idle(1083), False)
    expect("18:06 (내려 준 뒤)", veh.idle(1086), True)
    expect("01:00 (근무 끝)", veh.idle(1500), False)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

### 3.2 `build_costs` — 대기 승객 × 빈 차 비용행렬

10장의 비용행렬을 우리 객체로 만듭니다. 행이 승객, 열이 차량, 칸이 차가 승객에게 가는 분입니다.
`travel_time(차 위치, 승객 출발지, 지금 시각)` 이 칸 하나의 값입니다.

In [ ]:
banner("3.2 build_costs")
try:
    reqs = [sol.Request(0, (37.540, 127.200), (37.550, 127.210), 1080),
            sol.Request(1, (37.541, 127.201), (37.560, 127.220), 1080)]
    fleet = [sol.Vehicle(0, (37.539, 127.199), 1080, 1440),
             sol.Vehicle(1, (37.560, 127.230), 1080, 1440)]
    costs = sol.build_costs(reqs, fleet, 1080, straight_line_time)
    expect("행렬 크기", tuple(costs.shape), (2, 2))
    expect("costs[0, 0] (차 0 → 승객 0)", round(float(costs[0, 0]), 4), 0.3406, tol=1e-3)
    expect("costs[0, 1] (차 1 → 승객 0)", round(float(costs[0, 1]), 4), 8.2932, tol=1e-3)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

### 3.3 `assign` — 배차를 확정하고 차량 상태를 갱신

차가 승객에게 가는 시간, 승차 1분, 목적지까지 가는 시간, 하차 1분을 더한 시각이 차량의 `free_at` 이 됩니다.
차량은 승객을 내려 준 자리에 머뭅니다. `veh.location` 을 바꾸기 전에 공차 거리를 재야 합니다.

승객 0에게 차량 0을 보내면 탑승 1081.34, 하차 1085.75 가 나와야 합니다. 2절의 정답과 같은 값입니다.

In [ ]:
banner("3.3 assign")
try:
    req = sol.Request(0, (37.540, 127.200), (37.550, 127.210), 1080)
    veh = sol.Vehicle(0, (37.539, 127.199), 1080, 1440)
    pickup_min = straight_line_time(veh.location, req.origin, 1080)      # 0.34분
    sol.assign(req, veh, 1080, pickup_min, straight_line_time)

    expect("배차 시각", req.assigned_time, 1080)
    expect("탑승 시각", round(req.pickup_time, 2), 1081.34, tol=0.01)
    expect("하차 시각", round(req.dropoff_time, 2), 1085.75, tol=0.01)
    expect("차량 free_at = 하차 시각", veh.free_at, req.dropoff_time)
    expect("차량 위치 = 목적지", veh.location, req.dest)
    expect("공차 거리 (km)", round(veh.empty_km, 2), 0.14, tol=0.01)
    expect("일한 시간 (분)", round(veh.busy_min, 2), 5.75, tol=0.01)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

### 3.4 `simulate` — 1분 루프

DataFrame 을 `Request` 와 `Vehicle` 로 바꾼 뒤 1절의 다섯 단계를 매 분 반복합니다.
기록 컬럼 다섯 개는 DTUMOS 의 `record.csv` 와 이름이 같아야 합니다. 같은 형식이라야 6절에서 대조할 수 있습니다.

2절의 작은 예제로 돌려 정답과 같은 표가 나오는지 봅니다.

In [ ]:
banner("3.4 simulate (승객 3명, 차 2대, 10분)")
toy_mine = None
try:
    toy_mine = sol.simulate(toy_demand, toy_vehicles, 1080, 1090, travel_time=straight_line_time)
    expect("record 컬럼", list(toy_mine.record.columns), list(toy_answer.record.columns))
    expect("행 수", len(toy_mine.record), 10)
    expect("대기 승객 시계열", toy_mine.record["waiting_passenger_cnt"].tolist(),
           toy_answer.record["waiting_passenger_cnt"].tolist())
    expect("운행 차량 시계열", toy_mine.record["driving_vehicle_cnt"].tolist(),
           toy_answer.record["driving_vehicle_cnt"].tolist())
    expect("승객 2 의 배차 시각", toy_mine.requests[2].assigned_time, 1086)
    expect("config 에 fail_after_min", toy_mine.config.get("fail_after_min"), 10)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

### 3.5 `SimResult.summary` — 지표

채점과 비교에 쓰는 지표를 사전으로 냅니다. 열쇠 이름은 docstring 에 있습니다.
`avg_waiting_time_min` 은 배차받은 승객의 `wait_min` 평균이고,
`avg_assign_wait_min` 과 `avg_pickup_travel_min` 을 더하면 그 값이 됩니다.

In [ ]:
banner("3.5 summary (작은 예제)")
try:
    if toy_mine is None:
        raise NotImplementedError("simulate 가 아직 비어 있습니다")
    s = toy_mine.summary()
    for key in ("total_passengers", "served_passengers", "failed_passengers", "service_rate",
                "avg_waiting_time_min", "max_waiting_time_min", "avg_assign_wait_min",
                "avg_pickup_travel_min", "utilization", "empty_km", "loaded_km"):
        if key not in s:
            print(f"[x] 열쇠 {key} 가 없습니다")
    expect("배차받은 승객", s["served_passengers"], 3)
    expect("평균 대기 (분)", round(s["avg_waiting_time_min"], 2), 5.33, tol=0.01)
    expect("호출 → 배차 평균 (분)", round(s["avg_assign_wait_min"], 2), 1.0, tol=0.01)
    expect("배차 → 도착 평균 (분)", round(s["avg_pickup_travel_min"], 2), 4.33, tol=0.01)
    expect("가동률 (31분 / 720분)", round(s["utilization"], 3), 0.043, tol=0.001)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 4. 1시간만 돌려 봅니다 (교재 11.4)

작은 예제가 맞으면 하남 수요로 갑니다. 6시간을 한 번에 돌리지 말고 18:00~19:00 한 시간만 돌립니다.
틀렸을 때 어디가 틀렸는지 보이는 크기에서 시작합니다.

In [ ]:
banner("18:00 ~ 19:00 (60분)")
try:
    small = sol.simulate(demand, vehicles, 1080, 1140)
    display(small.record.head())
    s = small.summary()
    expect("호출 수", s["total_passengers"], 182)
    print(f"    배차 {s['served_passengers']}명, 평균 대기 {s['avg_waiting_time_min']:.2f}분")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

18:00 에 빈 차 80대로 시작해 1분 뒤 5대, 2분 뒤 9대가 호출을 받아 움직입니다.
한 시간 동안 호출 182건 중 181건이 배차되고 마지막 한 건은 19:00 에 아직 대기 중입니다.

## 5. 전체 구간 (교재 11.4)

18:00 부터 자정까지 360분입니다. 교재의 정돈본은 1초가 안 걸립니다. 몇 초를 넘으면 루프 안에서 불필요한 일을 하고 있는 것입니다.

In [ ]:
import time

banner("18:00 ~ 24:00 (360분)")
run = None
try:
    t0 = time.perf_counter()
    run = sol.simulate(demand, vehicles, 1080, 1440)
    print(f"    실행 {time.perf_counter() - t0:.2f}초")

    expect("record 행 수", len(run.record), 360)
    expect("record 컬럼", list(run.record.columns),
           ["time", "waiting_passenger_cnt", "fail_passenger_cnt",
            "empty_vehicle_cnt", "driving_vehicle_cnt"])

    s = run.summary()
    expect("서비스율", round(s["service_rate"], 3), 1.0, tol=0.02)
    expect("평균 대기 (분, 교재 4.28)", round(s["avg_waiting_time_min"], 2), 4.28, tol=0.5)
    print(f"    최대 대기 {s['max_waiting_time_min']:.1f}분, 가동률 {s['utilization']:.3f}")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

1,000명 전원 배차, 평균 대기 4.28분, 가동률 0.67이 교재의 값입니다.

## 6. 엔진과 맞춰 보기 (교재 11.5)

같은 도시, 같은 수요, 같은 차량으로 DTUMOS 를 돌린 결과가 저장소에 녹화되어 있습니다.
서버가 없어도 `data/fixtures/` 에서 읽어 옵니다.

In [ ]:
from smartmob import Dtumos

engine = Dtumos().run_simulation(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)
engine.summary()

엔진은 990명, 평균 대기 4.08분, 가동률 0.27입니다.
승객 수가 다른 것은 엔진이 자정 직전 열 건을 세지 않기 때문입니다.

`kpi_table` 은 두 결과를 같은 이름의 지표로 냅니다. 나란히 놓고 운행 차량 시계열의 상관계수도 잽니다.

In [ ]:
import numpy as np

from smartmob.teaching.metrics import kpi_table

try:
    if run is None:
        raise NotImplementedError("simulate 가 아직 비어 있습니다")
    table = pd.DataFrame({"내 루프": kpi_table(run), "DTUMOS 엔진": kpi_table(engine)})
    display(table.loc[["service_rate", "wait_mean", "wait_p50", "wait_p90", "wait_max",
                       "utilization", "empty_share"]].round(3))

    corr = np.corrcoef(run.record["driving_vehicle_cnt"], engine.record["driving_vehicle_cnt"])[0, 1]
    expect("운행 차량 시계열 상관계수 (0.8 이상)", round(corr, 3), 0.915, tol=0.115)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

평균 대기 4.28분 대 4.08분, 0.2분 차이입니다. 채점 기준은 이 차이 1.5분 이내입니다.

가동률은 같은 줄에 놓고 읽으면 안 됩니다.
엔진의 `utilization`(0.27)은 승객을 태운 시간만 셉니다.
우리 `summary()`(0.67)는 태우러 가는 시간까지 일한 시간으로 셉니다.
`kpi_table` 의 `utilization` 은 셋째 정의로, 분 단위 기록에서 (운행 중 차량 ÷ 근무 차량)의 평균입니다.
정의가 다른 지표를 나란히 놓으면 틀린 결론이 나옵니다.

값이 완전히 같지 않은 이유는 크게 셋입니다.

1. 소요시간 모형이 다릅니다. 내 루프는 직선거리 ÷ 25km/h 이고 엔진은 도로망 위를 달립니다
2. 엔진은 승객 수를 990명으로 셉니다
3. 엔진은 차량이 이동하는 동안의 위치를 도로 위에서 추적합니다

숫자가 다른 것 자체는 문제가 아닙니다. 왜 다른지 모르는 것이 문제입니다.

## 7. 대기시간은 두 부분입니다 (교재 11.6)

`fail_after_min` 은 10분입니다. 그런데 최대 대기가 26분입니다. 모순처럼 보입니다.
대기시간이 호출 → 배차 확정(`assign_wait_min`)과 배차 → 차 도착(`pickup_travel_min`)으로 나뉘기 때문입니다.
포기 기준은 앞부분에만 걸립니다.

In [ ]:
banner("대기시간 두 부분")
try:
    if run is None:
        raise NotImplementedError("simulate 가 아직 비어 있습니다")
    s = run.summary()
    print(f"    호출 → 배차 확정  평균 {s['avg_assign_wait_min']:.2f}분")
    print(f"    배차 → 차 도착    평균 {s['avg_pickup_travel_min']:.2f}분")
    print(f"    합계              평균 {s['avg_waiting_time_min']:.2f}분")

    over = [r for r in run.requests if r.wait_min is not None and r.wait_min > run.config["fail_after_min"]]
    worst = max(over, key=lambda r: r.wait_min)
    print(f"\n    총 대기가 10분을 넘은 승객 {len(over)}명")
    print(f"    최악: 배차까지 {worst.assign_wait_min:.0f}분 + 차 오는 데 {worst.pickup_travel_min:.1f}분")
    expect("배차 대기가 포기 기준을 넘는 승객", sum(1 for r in run.requests
           if r.assign_wait_min is not None and r.assign_wait_min >= 10), 0)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

차 80대가 남아돌아 배차는 거의 즉시 되고, 대기시간 4.28분은 대부분 차가 오는 시간입니다.
멀리 있는 차가 배차되면 총 대기가 26분까지 늘어나지만, 배차까지의 시간이 10분을 넘은 승객은 없습니다.
"평균 대기 4분"이라는 보고를 받으면 어느 대기인지 물어야 합니다.

## 8. 채점

`labs/check.py` 가 보는 것은 `tests/test_simloop.py` 가 보는 것과 같습니다. 여기를 통과하면 과제 채점도 통과합니다.

In [ ]:
from check import check

try:
    report = check("ch11")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)

## 9. 조건을 바꿔 보기 (교재 11.7)

루프가 있으면 "만약에"를 물을 수 있습니다. 1장의 질문으로 돌아갑니다. 차량을 줄이면 어떻게 될까요.

여러분의 `simulate` 가 완성되어 있으면 그것으로, 아직이면 교재의 정돈본으로 돌립니다.

In [ ]:
try:
    sol.simulate(toy_demand, toy_vehicles, 1080, 1090)
    simulate = sol.simulate
    print("여러분의 simulate 로 돌립니다")
except NotImplementedError:
    simulate = reference
    print("아직 빈칸이 있어 교재의 정돈본으로 돌립니다")

rows = []
for n in (20, 40, 60, 80):
    s = simulate(demand, vehicles.head(n), 1080, 1440).summary()     # head(n): 앞의 n대만 씁니다
    rows.append({
        "차량": n,
        "서비스율": round(s["service_rate"], 3),
        "평균대기(분)": round(s["avg_waiting_time_min"], 2),
        "최대대기(분)": round(s["max_waiting_time_min"], 1),
        "가동률": round(s["utilization"], 3),
        "공차(km)": s["empty_km"],
    })
sweep = pd.DataFrame(rows)
sweep

80대에서 40대로 줄이면 가동률이 0.67에서 0.98로 오르지만 서비스율이 1.0에서 0.72로 떨어집니다.
20대에서는 39%만 배차받습니다. 평균 대기가 7.3분으로 그대로인 것처럼 보이는 이유는 12장에서 봅니다.

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(sweep["차량"], sweep["평균대기(분)"], "o-", color="tab:red", label="평균 대기")
ax1.set_xlabel("차량 대수"); ax1.set_ylabel("평균 대기 (분)", color="tab:red")
ax1.tick_params(axis="y", labelcolor="tab:red")

ax2 = ax1.twinx()                # 같은 x축에 오른쪽 y축을 하나 더 만듭니다
ax2.plot(sweep["차량"], sweep["서비스율"], "s--", color="tab:blue", label="서비스율")
ax2.set_ylabel("서비스율", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(alpha=0.25, linewidth=0.6)
fig.tight_layout();

어느 지점을 고를지는 이 그림이 정해 주지 않습니다.
승객의 대기시간과 운영자의 차량 비용 중 무엇을 얼마나 중히 볼지는 사람이 정합니다.

## 10. 배차 방법을 바꿔 보기 (교재 11.8)

10장에서 헝가리안이 탐욕보다 낫다고 했습니다. 시뮬레이션 전체에서도 그럴까요.
`match="greedy"` 로 바꿔 80대와 25대에서 비교합니다.

In [ ]:
banner("optimal vs greedy")
for n in (80, 25):
    for method in ("optimal", "greedy"):
        s = simulate(demand, vehicles.head(n), 1080, 1440, match=method).summary()
        print(f"차량 {n:2d}대 {method:8s} 평균대기 {s['avg_waiting_time_min']:6.2f}분  "
              f"서비스율 {s['service_rate']:.3f}  공차 {s['empty_km']:7.1f}km")

80대에서는 4.28분 대 4.28분으로 차이가 없습니다. 대부분의 분에 대기 승객이 0~1명이라 짝지을 것이 하나뿐이기 때문입니다.
25대에서는 7.45분 대 20.9분입니다. 배차 알고리즘은 수요가 공급을 압박할 때만 의미가 있습니다.

## 제출할 것

1. 채운 `labs/ch11_simloop.py`
2. 8절 채점 셀의 출력 (전부 PASS)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄
4. 내 루프와 엔진이 왜 완전히 같지 않은지 한 문단

4번의 배점이 가장 큽니다. 숫자가 다른 것 자체는 문제가 아니고, 왜 다른지 모르는 것이 문제입니다.

## 정리

- 한 스텝은 다섯 가지입니다. 호출 접수 → 포기 처리 → 배차 → 차량 상태 갱신 → 기록
- 차량 상태는 `location` 과 `free_at` 두 값이면 충분합니다
- 작은 예제(승객 3명, 차 2대)에서 세 번째 호출은 배차까지 3분 기다립니다. 여기가 맞아야 전체가 맞습니다
- 하남 6시간이 1초 안에 끝납니다. 평균 대기 4.28분, 엔진 4.08분
- 대기시간은 배차까지와 차가 오는 동안으로 나뉩니다. 포기 기준은 앞부분에만 걸립니다
- 배차 알고리즘의 차이는 25대처럼 차가 모자랄 때만 드러납니다
- 12장 실습에서 이 결과를 보고서에 쓸 표와 그림으로 만듭니다